### Multi-Head Self-Attention Implementation
In this section, we initialize the input tensor and the linear layers for Queries, Keys, and Values.

In [58]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [59]:
B,T,embd =1,3,8

x = torch.randn(B,T,embd)

### Query, Key, and Value Projections
We define three separate linear transformations. In the attention mechanism:
- **Queries (Q)**: Represent what the token is 'looking for'.
- **Keys (K)**: Represent what the token 'contains' to match against queries.
- **Values (V)**: Represent the information that will be passed forward if a match is found.

In [60]:
Wq = nn.Linear(embd,embd,bias=False)
Wk = nn.Linear(embd,embd,bias=False)
Wv = nn.Linear(embd,embd,bias=False)

In [61]:
q = Wq(x)
k = Wk(x)
v = Wv(x)

In [62]:
q.shape,k.shape,v.shape

(torch.Size([1, 3, 8]), torch.Size([1, 3, 8]), torch.Size([1, 3, 8]))

### Reshaping for Multi-Head Attention
We split the embedding dimension across multiple heads to allow the model to attend to information from different representation subspaces.

In [63]:
n_heads = 4
head_dim = embd // n_heads

In [64]:
q = q.view(B,T,n_heads,head_dim).transpose(1,2)
q.view(B,n_heads,T,head_dim)

tensor([[[[ 0.0044, -0.2547],
          [-0.8612,  0.6625],
          [-0.8049,  0.3799]],

         [[-1.0368,  1.2336],
          [ 0.8799,  0.6102],
          [ 1.1122, -0.1207]],

         [[ 0.7675, -1.4310],
          [ 0.1485,  0.1051],
          [-0.4221, -0.0971]],

         [[ 0.4554, -0.7085],
          [-1.0178,  0.5353],
          [-1.2816, -0.7058]]]], grad_fn=<ViewBackward0>)

In [65]:
k = k.view(B,T,n_heads,head_dim).transpose(1,2)
v = v.view(B,T,n_heads,head_dim).transpose(1,2)

### Scaled Dot-Product Attention
We calculate the raw attention scores by performing a batch matrix multiplication between Queries and Keys ($Q K^T$).

We then scale the scores by $\frac{1}{\sqrt{d_k}}$ to prevent the dot product from growing too large in magnitude, which can lead to vanishing gradients during softmax.

In [66]:
scores = q @ k.transpose(-2,-1)
print(scores)

print(scores.shape)

tensor([[[[-0.4033, -0.0668, -0.0373],
          [ 1.3337, -0.0073, -0.7655],
          [ 0.8689, -0.0705, -0.7547]],

         [[-0.5929, -1.6885, -0.5685],
          [-0.7588, -0.3036, -0.0840],
          [-0.2798,  0.5510,  0.1987]],

         [[-1.8186,  0.1023,  0.8150],
          [ 0.0899,  0.1615, -0.0687],
          [-0.0222, -0.3843,  0.0758]],

         [[ 0.1739, -0.1178,  0.1634],
          [-0.0525,  0.4701,  0.0263],
          [ 0.3767,  0.8641,  0.5484]]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([1, 4, 3, 3])


In [67]:
scale = 1 /math.sqrt(embd)
scores = scores*scale
print(scores)

tensor([[[[-0.1426, -0.0236, -0.0132],
          [ 0.4715, -0.0026, -0.2707],
          [ 0.3072, -0.0249, -0.2668]],

         [[-0.2096, -0.5970, -0.2010],
          [-0.2683, -0.1073, -0.0297],
          [-0.0989,  0.1948,  0.0703]],

         [[-0.6430,  0.0362,  0.2882],
          [ 0.0318,  0.0571, -0.0243],
          [-0.0078, -0.1359,  0.0268]],

         [[ 0.0615, -0.0417,  0.0578],
          [-0.0185,  0.1662,  0.0093],
          [ 0.1332,  0.3055,  0.1939]]]], grad_fn=<MulBackward0>)


### Causal Masking
We apply a lower triangular mask to ensure that the attention mechanism only attends to past and current tokens, not future ones (Auto-regressive property).

In [68]:
mask = torch.tril(torch.ones(T,T).view(1,T,T))
scores = scores.masked_fill(mask==0,float('-inf'))

In [69]:
att = F.softmax(scores,dim=-1)

In [70]:
y = att @ v
y.shape

torch.Size([1, 4, 3, 2])

### Concatenation and Final Projection
After calculating attention for each head, we concatenate the results back into the original embedding dimension. A final linear layer is applied to allow the heads to interact with each other.

In [71]:
y = y.transpose(1,2).contiguous().view(B,T,embd)
y.shape

torch.Size([1, 3, 8])

### Output Projection and Logits Generation
Finally, we project the concatenated head outputs back to the embedding dimension and map them to our vocabulary size to get logits.

In [72]:
map = nn.Linear(embd,embd,bias=False)
ys = map(y)
ys.shape

torch.Size([1, 3, 8])

In [73]:
vocab_size = 10
vocab = nn.Linear(embd,vocab_size,bias=False)
logits = vocab(y)
logits.shape

torch.Size([1, 3, 10])

In [74]:
probs = F.softmax(logits,dim=-1)
probs.shape

torch.Size([1, 3, 10])

In [75]:
next_token = probs[:,-1,:]
next_token

tensor([[0.1355, 0.0678, 0.0863, 0.0954, 0.0657, 0.1063, 0.0908, 0.1059, 0.1222,
         0.1240]], grad_fn=<SelectBackward0>)

### Decoding Strategies
Once we have the probability distribution for the next token, we need a strategy to pick the actual token ID. We will explore **Greedy Search**, **Top-K Sampling**, and **Temperature Scaling**.

In [76]:
## greedy sampling (detrminsitic,fast and pick token with high prob)
## classification,factual content,rag
idx = torch.argmax(next_token,dim=-1)
idx.shape

torch.Size([1])

In [77]:
## top-k sampling (limit randomness,prevents low porb nonsense)

topk_probs,topk_idxs = torch.topk(next_token,5,dim=-1)
topk_probs,topk_idxs



(tensor([[0.1355, 0.1240, 0.1222, 0.1063, 0.1059]], grad_fn=<TopkBackward0>),
 tensor([[0, 9, 8, 5, 7]]))

In [78]:
idx = torch.multinomial(topk_probs,1)
idx

tensor([[4]])

In [79]:
x_col = torch.gather(topk_idxs,dim=-1,index=idx)
x_col

tensor([[7]])

### Understanding Temperature Scaling

Temperature scaling modifies the **logits** (the raw outputs before softmax) using the formula:

$$P_i = \frac{\exp(z_i / T)}{\sum_{j} \exp(z_j / T)}$$

Where $z$ are the logits and $T$ is the temperature.

#### 1. High Temperature ($T > 1$)
- As $T \to \infty$, the distribution becomes **uniform**.
- Differences between logits are suppressed, making low-probability tokens more likely to be sampled.
- **Result:** More creative, diverse, but potentially nonsensical outputs.

#### 2. Low Temperature ($T < 1$)
- As $T \to 0$, the distribution becomes **sharper**.
- The gap between the highest logit and the others is amplified.
- **Result:** More confident, conservative, and deterministic outputs. A temperature of $T \to 0$ is equivalent to **Greedy Search**.

#### 3. Neutral Temperature ($T = 1$)
- The distribution remains exactly as output by the model's softmax layer.

In [80]:
temperature = 0.7

# Apply temperature to logits before softmax
scaled_logits = logits[:,-1,:] / temperature
scaled_probs = F.softmax(scaled_logits, dim=-1)

print(f"Original Probs: {next_token[0]}")
print(f"Scaled Probs (T={temperature}): {scaled_probs[0]}")

# Sample from the scaled distribution
idx_temp = torch.multinomial(scaled_probs, 1)
print(f"Sampled index with temperature: {idx_temp.item()}")

Original Probs: tensor([0.1355, 0.0678, 0.0863, 0.0954, 0.0657, 0.1063, 0.0908, 0.1059, 0.1222,
        0.1240], grad_fn=<SelectBackward0>)
Scaled Probs (T=0.7): tensor([0.1521, 0.0566, 0.0799, 0.0921, 0.0541, 0.1074, 0.0858, 0.1069, 0.1312,
        0.1339], grad_fn=<SelectBackward0>)
Sampled index with temperature: 0
